# 01 — Test vol-engine (état container + healthcheck + env vars)

Smoke test du container `fxvol-vol-engine` (service `vol-engine` côté compose) — étape 1/5. Valide les **fondations** avant de tester les outputs Redis (surface SVI/SSVI + signaux), le pub/sub et le bridge WS dans les notebooks suivants : container UP, healthcheck heartbeat-based passe `healthy`, env vars critiques (incl. les 2 calibrations vol-spécifiques) correctement injectées.

**Couvre** :
1. Container présent et `running`
2. Docker healthcheck = `healthy` (= `heartbeat:vol_engine` < 300s en Redis, cf. `docker-compose.yml § vol-engine`)
3. Image attendue (`fx-options-vol-engine:local`) + `StartedAt` + restart count raisonnable
4. Env vars critiques : `SERVICE_NAME=vol_engine`, `IB_HOST=ib-gateway`, `IB_PORT=4002`, `IB_CLIENT_ID=2`, `REDIS_URL=redis://redis:6379/0`, `MODEL_P` (har|garch), `THRESHOLD_VOL_PTS` (float)

**Préreq** :
- Stack démarrée avec profile `engines` + `ib` : `docker compose --profile engines --profile ib up -d`
- `redis` healthy (le healthcheck de vol-engine dépend de Redis pour vérifier le heartbeat)
- `market-data` started (dependency listée dans compose ; sans spot publié, vol-engine cycle skip mais pousse quand même le heartbeat)
- `ib-gateway` started (la connexion IB est ouverte et utilisée pour reqMktData chains + reqHistoricalData OHLC ; si ib-gateway down, l'engine retry en backoff sans bloquer)
- **~4-5 min d'attente** après `up` pour que `start_period` (240s) soit fini et qu'au moins 1 cycle complet ait poussé un heartbeat. Vol-engine est plus lent à démarrer que risk (cycle 30-180s + GARCH/SVI/SSVI fits = 2-5s de CPU).

**Référence** : `docker-compose.yml § vol-engine`, `src/services/vol/main.py`, `src/services/vol/engine.py`.

## Setup

Note : le container Docker s'appelle `fxvol-vol-engine` (avec `-engine`), contrairement à `fxvol-risk` (sans `-engine`) et `fxvol-market-data` (sans `-engine`). Naming hétérogène volontairement non corrigé pour ne pas casser les volumes / scripts existants — à harmoniser plus tard si besoin.

In [101]:
import subprocess

CONTAINER = "fxvol-vol-engine"
results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

def docker_inspect(fmt):
    out = subprocess.run(
        ["docker", "inspect", "-f", fmt, CONTAINER],
        capture_output=True, text=True,
    )
    return out.stdout.strip()

print(f"target container = {CONTAINER}")

target container = fxvol-vol-engine


## 1. Container présent et `running`

**Ce que tu dois voir** : state = `running`. Si `exited` → crash au boot (`docker logs fxvol-vol-engine --tail 100`). Si `restarting` → crashloop. Si `<not found>` → stack pas démarrée avec profile `engines`.

In [102]:
print("== 1. container state ==")

state = docker_inspect("{{.State.Status}}")
record("docker container state", state == "running", state or "<not found>")

restart_count = docker_inspect("{{.RestartCount}}")
record("restart count raisonnable (≤ 3)",
       restart_count.isdigit() and int(restart_count) <= 3,
       f"restart count = {restart_count}")

== 1. container state ==
  [OK  ] docker container state  | running
  [OK  ] restart count raisonnable (≤ 3)  | restart count = 0


## 2. Docker healthcheck = `healthy`

Le healthcheck (parsing ISO-8601, fix R9) lit `heartbeat:vol_engine` dans Redis et vérifie que l'âge est **< 300s** (vs 30s pour risk-engine). Cette tolérance large vient du fait que vol-engine fait des calculs lourds par cycle (GARCH/HAR/SVI/SSVI fits sur 80+ strikes) et que le cycle nominal peut atteindre 30-60s. 300s = 10 cycles nominaux, la HC ne flap pas si un cycle est ralenti par OHLC lent.

`start_period=240s` (4 min) : le 1er cycle paye fetch OHLC 1Y + GARCH fit + SVI par tenor + SSVI surface ; sur une cold start machine modeste c'est ~3 min. Avant 4 min, HC = `starting` (pas `unhealthy`), c'est attendu.

In [103]:
print("== 2. healthcheck ==")

health = docker_inspect("{{.State.Health.Status}}")
record("docker healthcheck", health == "healthy", health or "<no healthcheck>")

last_log = docker_inspect("{{(index .State.Health.Log 0).Output}}")
if last_log:
    print(f"  [INFO] dernière sortie healthcheck : {last_log[:200]!r}")

== 2. healthcheck ==
  [OK  ] docker healthcheck  | healthy


## 3. Image + StartedAt (info)

Pas un test pass/fail — juste de l'info pour le diag.

In [104]:
print("== 3. image + uptime info ==")

image = docker_inspect("{{.Config.Image}}")
started_at = docker_inspect("{{.State.StartedAt}}")
print(f"  [INFO] image      : {image}")
print(f"  [INFO] StartedAt  : {started_at}")

expected_prefix = "fx-options-vol-engine"
record(f"image préfixe '{expected_prefix}'",
       expected_prefix in image,
       image)

== 3. image + uptime info ==
  [INFO] image      : fx-options-vol-engine:local
  [INFO] StartedAt  : 2026-04-29T09:19:44.124546731Z
  [OK  ] image préfixe 'fx-options-vol-engine'  | fx-options-vol-engine:local


## 4. Env vars critiques injectées

Filtre positif explicite (cf. mémoire `feedback_docker_inspect_env_leak`). Les 7 vars listées sont toutes non-secrètes, afficher leurs valeurs ne pose pas de risque.

**ClientId 2 critique** : doit être différent de market-data (1) et risk-engine (3). Si override accidentel sur 1 ou 3 → kick mutuel sur Gateway (erreur 326 "client id already in use") et l'un des 2 engines crashe en boucle.

**Vars vol-spécifiques** :
- `MODEL_P` ∈ {`har`, `garch`} : choix de l'estimateur P-measure pour le fair value. Default `har` (HAR-RV de Corsi 2009, plus robuste sur FX intraday).
- `THRESHOLD_VOL_PTS` (float, default `1.0`) : largeur de la zone FAIR autour du fair value, en points de vol. Si `|sigma_mid - sigma_fair| < THRESHOLD` → signal FAIR ; au-dessus → CHEAP/EXPENSIVE.

In [105]:
print("== 4. env vars critiques (filtrées non-secret) ==")

expected = {
    "SERVICE_NAME": "vol_engine",
    "IB_HOST": "ib-gateway",
    "IB_PORT": "4002",
    "IB_CLIENT_ID": "2",
    "REDIS_URL": "redis://redis:6379/0",
}
# Ces 2-là ont une valeur dynamique (default ou override), on ne fixe que la présence.
soft_expected = {"MODEL_P", "THRESHOLD_VOL_PTS"}

all_keys = list(expected) + list(soft_expected)
key_pattern = "^(" + "|".join(all_keys) + ")="
out = subprocess.run(
    ["docker", "exec", CONTAINER, "sh", "-c",
     f'env | grep -E "{key_pattern}"'],
    capture_output=True, text=True,
)
lines = out.stdout.strip().splitlines()
actual = dict(line.split("=", 1) for line in lines if "=" in line)

for key, expected_val in expected.items():
    actual_val = actual.get(key, "<missing>")
    record(f"{key} = {expected_val!r}",
           actual_val == expected_val,
           actual_val if actual_val != expected_val else "")

for key in soft_expected:
    actual_val = actual.get(key, "<missing>")
    present = actual_val != "<missing>"
    record(f"{key} présente (valeur libre)",
           present,
           f"= {actual_val!r}" if present else "<missing>")

# Validation soft de MODEL_P : doit être dans {har, garch}
model_p = actual.get("MODEL_P", "").lower()
record("MODEL_P ∈ {har, garch}",
       model_p in {"har", "garch"},
       f"= {model_p!r}")

# Validation soft de THRESHOLD_VOL_PTS : doit être un float positif
try:
    thr = float(actual.get("THRESHOLD_VOL_PTS", ""))
    thr_ok = thr > 0
except ValueError:
    thr = None
    thr_ok = False
record("THRESHOLD_VOL_PTS float > 0", thr_ok, f"= {thr}")

== 4. env vars critiques (filtrées non-secret) ==
  [OK  ] SERVICE_NAME = 'vol_engine'
  [OK  ] IB_HOST = 'ib-gateway'
  [OK  ] IB_PORT = '4002'
  [OK  ] IB_CLIENT_ID = '2'
  [OK  ] REDIS_URL = 'redis://redis:6379/0'
  [OK  ] THRESHOLD_VOL_PTS présente (valeur libre)  | = '1.0'
  [OK  ] MODEL_P présente (valeur libre)  | = 'har'
  [OK  ] MODEL_P ∈ {har, garch}  | = 'har'
  [OK  ] THRESHOLD_VOL_PTS float > 0  | = 1.0


## Récap final

In [106]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 100)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 100)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK container vol-engine sain. Pass au notebook 02 (Redis outputs : surface SVI/SSVI, signaux, heartbeat).")


LABEL                                                   STATUS  DETAIL
----------------------------------------------------------------------------------------------------
docker container state                                  OK      running
restart count raisonnable (≤ 3)                         OK      restart count = 0
docker healthcheck                                      OK      healthy
image préfixe 'fx-options-vol-engine'                   OK      fx-options-vol-engine:local
SERVICE_NAME = 'vol_engine'                             OK      
IB_HOST = 'ib-gateway'                                  OK      
IB_PORT = '4002'                                        OK      
IB_CLIENT_ID = '2'                                      OK      
REDIS_URL = 'redis://redis:6379/0'                      OK      
THRESHOLD_VOL_PTS présente (valeur libre)               OK      = '1.0'
MODEL_P présente (valeur libre)                         OK      = 'har'
MODEL_P ∈ {har, garch}                  

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `state` = `<not found>` | Stack lancée sans profile `engines` | `docker compose --profile engines --profile ib up -d` |
| `state` = `restarting` ou restart count > 3 | Crashloop : engine plante au boot | `docker logs fxvol-vol-engine --tail 100` ; suspects : `IB_HOST` mauvais, Redis URL incorrecte, ib-gateway pas joignable au boot, `tzdata` manquant (cf. R9 fix), erreur d'import config vol |
| `healthcheck` = `starting` depuis < 4 min | start_period (240s) pas atteint | attendre 4 min ; le 1er cycle paye OHLC + GARCH + SVI/SSVI fits |
| `healthcheck` = `starting` depuis > 5 min | 1er cycle hang sur IB OHLC | `docker logs fxvol-vol-engine --tail 100` ; chercher `Fetching daily OHLC` qui ne termine pas → IB historical perms KO ou rate-limit |
| `healthcheck` = `unhealthy` | Heartbeat absent ou stale > 300s | `docker compose exec redis redis-cli GET heartbeat:vol_engine` ; si vide → engine planté en cycle, voir logs |
| `IB_CLIENT_ID != "2"` | Override compose appliqué | OK si voulu, sinon revoir compose ; risque collision avec market-data (1) ou risk (3) → erreur IB 326 |
| `MODEL_P` ∉ {`har`, `garch`} | Typo dans `.env` ou override CI | revoir env ; valeurs autres font crasher l'engine au boot |
| `THRESHOLD_VOL_PTS` non parsable | Typo string au lieu de float | revoir env ; default `1.0` si non set |
| `image` ≠ `fx-options-vol-engine:*` | Image custom (release pinnée, registry distant) | Pas un fail si voulu, sinon revoir build |